### Trabalho Fase 2 do Curso de Pós-Graduação FIAP IA para Devs
#### Parte 8 - Tuning de Regressão Logística com Algoritmos Genéticos

Fonte de dados escolhida: DATASUS/SISCAN  
Tipo de dados de origem nesta etapa: Parquet

Arquivos utilizados:

```text
bases/treino/x_encoded.parquet
bases/treino/y.parquet
bases/teste/x_encoded.parquet
bases/teste/y.parquet
```

---

## Objetivo da Parte 8

O objetivo desta etapa é otimizar os hiperparâmetros do modelo de Regressão Logística (`LogisticRegression`) usando um Algoritmo Genético implementado diretamente no notebook. A estrutura segue a Parte 7 como one-shot: mesma leitura de dados, mesma estratégia de amostragem, mesmos operadores genéticos e mesma função de fitness voltada ao desbalanceamento da target.

O cromossomo de cada indivíduo representa uma configuração da Regressão Logística:

- `C`;
- `class_weight`;
- `scaler_type`;
- `max_iter`.

Breve descrição dos hiperparâmetros otimizados:

- `C`: controla a intensidade da regularização da Regressão Logística. Valores menores aumentam a regularização e reduzem a complexidade do modelo; valores maiores reduzem a regularização e permitem maior ajuste aos dados. O objetivo do AG é buscar uma configuração que preserve generalização e melhore o desempenho na classe positiva.
- `class_weight`: define pesos diferenciados para as classes. A opção `balanced` aumenta a importância da classe minoritária durante o treino, o que é relevante porque a target possui poucos casos positivos. O objetivo é melhorar recall e F1 sem depender de accuracy isolada.
- `scaler_type`: define se as features serão usadas sem escala, com padronização ou com normalização. Como a Regressão Logística é sensível à escala das variáveis, esse gene avalia qual transformação favorece os coeficientes e a separação das classes.
- `max_iter`: define o número máximo de iterações do processo de otimização. Valores maiores ajudam o solver a convergir em configurações mais difíceis, mas aumentam o custo computacional. O objetivo é evitar convergência incompleta mantendo tempo de execução aceitável.

A coluna de idade mantida neste notebook é `CO_IDADE_PACIENTE_NUM`, alinhada ao melhor baseline de Regressão Logística da Parte 6.

---

## Índice / Sumário da Parte 8

**Item 1 - Leitura e preparação dos dados para Regressão Logística**

- Leitura das bases encoded geradas na Parte 2.
- Definição de constantes, espaços de busca e métricas usadas na avaliação.
- Seleção das colunas usadas pela Regressão Logística.

**Item 2 - Amostragem estratificada**

- Criação de amostras estratificadas de treino e teste.
- Preservação aproximada da proporção da classe positiva.

**Item 3 - Execução da Regressão Logística com validação cruzada**

- Aplicação opcional de escala.
- Treinamento do `LogisticRegression` em 5 folds estratificados.
- Retorno das métricas médias dos folds.

**Item 4 - Função de fitness**

- Cálculo da fitness com a combinação `0.7 * F1 + 0.3 * recall`.

**Item 5 - Representação genética e população inicial**

- Definição do cromossomo, solução base, variação inicial e criação dos indivíduos.

**Item 6 - Operadores genéticos**

- Avaliação da população.
- Seleção por torneio.
- Crossover uniforme.
- Mutação por gene.
- Elitismo e geração da nova população.

**Item 7 - Experimentos com Algoritmo Genético**

- Execução de três configurações de população, mutação e número de gerações.
- Registro do melhor indivíduo por geração.

**Item 8 - Avaliação final da Regressão Logística otimizada**

- Escolha do melhor indivíduo entre os três experimentos.
- Treinamento final sem validação cruzada.
- Avaliação na amostra estratificada de teste.

**Item 9 - Comparação com a Parte 6 e conclusão**

- Comparação entre o baseline de Regressão Logística da Parte 6 e a Regressão Logística otimizada por Algoritmo Genético.
- Síntese dos ganhos, perdas e trade-offs observados nas métricas da classe positiva.


#### Item 1 - Leitura e preparação dos dados para Regressão Logística

A Parte 8 usa diretamente `x_encoded.parquet`, criado na Parte 2. Como a validação das bases já foi realizada na etapa de preparação, este bloco concentra os imports, a configuração das métricas, a leitura das bases e a definição dos espaços de busca do Algoritmo Genético.

Também é definida a estrutura `ClassificationMetrics`, usada para padronizar as métricas calculadas ao longo do notebook.


In [1]:
from pathlib import Path

import random
import time
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, Normalizer
try:
    from IPython.display import display, HTML
except ImportError:
    def display(value):
        print(value)

    class HTML(str):
        pass
import numpy as np

from dataclasses import dataclass

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

random.seed(42)
np.random.seed(42)

display(HTML("""
<style>
.container {
    width: 100% !important;
    max-width: 100% !important;
}
.jp-Cell-outputArea,
.jp-OutputArea-output,
.output_area {
    width: 100% !important;
    max-width: 100% !important;
}
.dataframe {
    width: 100% !important;
}
.dataframe td,
.dataframe th {
    white-space: normal !important;
    word-break: break-word !important;
}
</style>
"""))

@dataclass
class ClassificationMetrics:
    accuracy: float
    recall: float
    f1: float
    precision: float
    roc_auc: float
    balanced_accuracy: float


def calculate_metrics(y_true, y_pred, y_score=None) -> ClassificationMetrics:
    roc_auc_input = y_score if y_score is not None else y_pred
    return ClassificationMetrics(
        accuracy=accuracy_score(y_true, y_pred),
        recall=recall_score(y_true, y_pred, zero_division=0),
        f1=f1_score(y_true, y_pred, zero_division=0),
        precision=precision_score(y_true, y_pred, zero_division=0),
        roc_auc=roc_auc_score(y_true, roc_auc_input),
        balanced_accuracy=balanced_accuracy_score(y_true, y_pred),
    )


C_SPACE = [0.01, 0.1, 1, 10, 100]
CLASS_WEIGHT_SPACE = [None, "balanced"]
SCALER_SPACE = [None, "standard", "normalizer"]
MAX_ITER_SPACE = [1000, 2000, 5000, 10000]

RANDOM_STATE = 42
TARGET_COLUMN = 'TARGET_CANCER_MAMA_PROVAVEL'
DATA_DIR = Path('bases')

N_TRAIN_LOGREG = 12000
N_VALIDATION_LOGREG = 4000
N_TEST_LOGREG = 8000

input_files = {
    'x_train_encoded': DATA_DIR / 'treino' / 'x_encoded.parquet',
    'y_train': DATA_DIR / 'treino' / 'y.parquet',
    'x_test_encoded': DATA_DIR / 'teste' / 'x_encoded.parquet',
    'y_test': DATA_DIR / 'teste' / 'y.parquet',
}

X_train_encoded = pd.read_parquet(input_files['x_train_encoded'])
y_train = pd.read_parquet(input_files['y_train'])[TARGET_COLUMN]
X_test_encoded = pd.read_parquet(input_files['x_test_encoded'])
y_test = pd.read_parquet(input_files['y_test'])[TARGET_COLUMN]

X_train_encoded = X_train_encoded.drop(columns=['CO_IDADE_PACIENTE_CAP'])
X_test_encoded = X_test_encoded.drop(columns=['CO_IDADE_PACIENTE_CAP'])

age_columns = ['CO_IDADE_PACIENTE_NUM']

context_numeric_columns = [
    column for column in ['CO_TEMPO_MAMO_ANTERIOR_NUM']
    if column in X_train_encoded.columns
]
encoded_categorical_columns = [
    column for column in X_train_encoded.columns
    if column not in age_columns + context_numeric_columns
]

logreg_input_summary = {
    'x_train_encoded_shape': X_train_encoded.shape,
    'x_test_encoded_shape': X_test_encoded.shape,
    'selected_age_columns': age_columns,
    'context_numeric_column_count': len(context_numeric_columns),
    'context_numeric_columns': context_numeric_columns,
    'encoded_categorical_column_count': len(encoded_categorical_columns),
}
logreg_input_summary


{'x_train_encoded_shape': (213978, 24),
 'x_test_encoded_shape': (53495, 24),
 'selected_age_columns': ['CO_IDADE_PACIENTE_NUM'],
 'context_numeric_column_count': 1,
 'context_numeric_columns': ['CO_TEMPO_MAMO_ANTERIOR_NUM'],
 'encoded_categorical_column_count': 22}

#### Item 2 - Amostragem estratificada

A amostragem estratificada preserva aproximadamente a proporção da target, que é fortemente desbalanceada. A amostra de treino é usada pelo Algoritmo Genético com validação cruzada; a amostra de teste fica reservada para a avaliação final do melhor indivíduo. Para manter a comparação com a Parte 6, a amostra final de teste usa `N_TEST_LOGREG = 8000` registros.


In [2]:
def limit_stratified_sample(X, y, sample_size, random_state=RANDOM_STATE):
    if sample_size is None or sample_size >= len(X):
        return X.copy(), y.copy()

    _, X_sample, _, y_sample = train_test_split(
        X,
        y,
        test_size=sample_size,
        random_state=random_state,
        stratify=y,
    )
    return X_sample.reset_index(drop=True), y_sample.reset_index(drop=True)


X_train_sample, y_train_sample = limit_stratified_sample(
    X_train_encoded,
    y_train,
    N_TRAIN_LOGREG + N_VALIDATION_LOGREG,
)

X_test_sample, y_test_sample = limit_stratified_sample(
    X_test_encoded,
    y_test,
    N_TEST_LOGREG,
)

logreg_samples_summary = pd.DataFrame([
    {'base': 'treino_logreg', 'linhas': len(X_train_sample), 'classe_positiva': int(y_train_sample.sum()), 'percentual_positivo': round(y_train_sample.mean() * 100, 2)},
    {'base': 'teste_logreg', 'linhas': len(X_test_sample), 'classe_positiva': int(y_test_sample.sum()), 'percentual_positivo': round(y_test_sample.mean() * 100, 2)},
])
logreg_samples_summary


,base,linhas,classe_positiva,percentual_positivo
0,treino_logreg,16000,679,4.24
1,teste_logreg,8000,339,4.24


#### Item 3 - Execução da Regressão Logística com validação cruzada

O bloco abaixo reúne a aplicação opcional de escala e a avaliação do `LogisticRegression`. Para cada indivíduo, o modelo é treinado em 5 folds estratificados, e as métricas finais retornadas são as médias dos folds.

Essa função é usada durante o Algoritmo Genético para avaliar cada combinação de hiperparâmetros sem usar o conjunto de teste.


In [3]:
def apply_scaler(x_train, x_test, scaler_type):
    if scaler_type == "standard":
        scaler = StandardScaler()
        X_train_processed = scaler.fit_transform(x_train)
        X_test_processed = scaler.transform(x_test)
        return X_train_processed, X_test_processed

    if scaler_type == "normalizer":
        scaler = Normalizer()
        X_train_processed = scaler.fit_transform(x_train)
        X_test_processed = scaler.transform(x_test)
        return X_train_processed, X_test_processed

    if scaler_type is None:
        return x_train, x_test

    raise ValueError("scaler_type deve ser 'standard', 'normalizer' ou None")


def execute_logistic_regression(
    x,
    y,
    C=1.0,
    class_weight=None,
    scaler_type="standard",
    max_iter=2000,
    show=False,
    short_show=False,
):
    n_splits = 5
    skf = StratifiedKFold(n_splits, shuffle=True, random_state=RANDOM_STATE)

    metrics = ClassificationMetrics(
        accuracy=0.0,
        recall=0.0,
        f1=0.0,
        precision=0.0,
        roc_auc=0.0,
        balanced_accuracy=0.0,
    )

    for train_idx, val_idx in skf.split(x, y):
        X_tr, X_val = x.iloc[train_idx], x.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        X_train_processed, X_test_processed = apply_scaler(X_tr, X_val, scaler_type)

        classifier = LogisticRegression(
            C=C,
            class_weight=class_weight,
            solver="liblinear",
            max_iter=max_iter,
            random_state=RANDOM_STATE,
        )

        classifier.fit(X_train_processed, y_tr)
        y_pred = classifier.predict(X_test_processed)
        y_score = classifier.predict_proba(X_test_processed)[:, 1]

        current_execution = calculate_metrics(y_val, y_pred, y_score)

        metrics.accuracy += current_execution.accuracy
        metrics.recall += current_execution.recall
        metrics.f1 += current_execution.f1
        metrics.precision += current_execution.precision
        metrics.roc_auc += current_execution.roc_auc
        metrics.balanced_accuracy += current_execution.balanced_accuracy

    metrics.accuracy /= n_splits
    metrics.recall /= n_splits
    metrics.f1 /= n_splits
    metrics.precision /= n_splits
    metrics.roc_auc /= n_splits
    metrics.balanced_accuracy /= n_splits

    if show:
        if short_show:
            print(f'recall: {metrics.recall} f1: {metrics.f1}')
        else:
            print(f'########### LogisticRegression ##############')
            print(f'C: {C}')
            print(f'class_weight: {class_weight}')
            print(f'solver: liblinear')
            print(f'scaler_type: {scaler_type}')
            print(f'max_iter: {max_iter}')
            print(f'accuracy: {metrics.accuracy}')
            print(f'recall: {metrics.recall}')
            print(f'f1: {metrics.f1}')
            print(f'precision: {metrics.precision}')
            print(f'roc_auc: {metrics.roc_auc}')
            print(f'balanced_accuracy: {metrics.balanced_accuracy}')
            print('#############################################')

    return metrics


#### Item 4 - Função de fitness

A fitness prioriza a classe positiva, que é minoritária. A métrica principal combina F1 e recall:

```text
fitness = 0.7 * F1 + 0.3 * recall
```

O F1 equilibra precision e recall, enquanto o recall aumenta a importância de recuperar casos positivos.


In [4]:
def fitness(individual, show=False, short_show=False) -> float:
    logreg = execute_logistic_regression(
        x=X_train_sample,
        y=y_train_sample,
        C=individual["C"],
        class_weight=individual["class_weight"],
        scaler_type=individual["scaler_type"],
        max_iter=individual["max_iter"],
        show=show,
        short_show=short_show,
    )

    return calculate_logreg_execution_fitness(logreg)


def calculate_logreg_execution_fitness(logreg_execution):
    return 0.7 * logreg_execution.f1 + 0.3 * logreg_execution.recall


#### Item 5 - Representação genética e população inicial

Cada indivíduo representa um conjunto de hiperparâmetros da Regressão Logística. A população inicial é criada a partir de uma solução base com variações aleatórias, estratégia usada como hot start para iniciar a busca em uma região plausível do espaço de hiperparâmetros.


In [5]:
def create_random_individuals(individual_count):
    return [{
        "C": random.choice(C_SPACE),
        "class_weight": random.choice(CLASS_WEIGHT_SPACE),
        "scaler_type": random.choice(SCALER_SPACE),
        "max_iter": random.choice(MAX_ITER_SPACE),
    } for _ in range(individual_count)]


BASE_LOGREG_SOLUTION = {
    "C": 1.0,
    "class_weight": "balanced",
    "scaler_type": "standard",
    "max_iter": 2000,
}

VARIATION_RATE = 0.3


def create_logreg_variation(base_solution, variation_rate):
    individual = base_solution.copy()

    if random.random() < variation_rate:
        individual["C"] = random.choice(C_SPACE)

    if random.random() < variation_rate:
        individual["class_weight"] = random.choice(CLASS_WEIGHT_SPACE)

    if random.random() < variation_rate:
        individual["scaler_type"] = random.choice(SCALER_SPACE)

    if random.random() < variation_rate:
        individual["max_iter"] = random.choice(MAX_ITER_SPACE)

    return individual


def create_hot_start_individuals(individual_count):
    return [create_logreg_variation(BASE_LOGREG_SOLUTION, VARIATION_RATE) for _ in range(individual_count)]


#### Item 6.1 - Avaliação da população

Cada indivíduo recebe uma nota de fitness calculada pela validação cruzada. Em seguida, a população é ordenada da maior para a menor fitness, permitindo selecionar os melhores candidatos para reprodução e elitismo.


In [6]:
def evaluate_population(individuals):   
    for individual in individuals:
        individual['evaluation'] = fitness(individual)

    sorted_population = sorted(
        individuals,
        key=lambda item: item["evaluation"],
        reverse=True
    )
    
    return sorted_population           


#### Item 6.2 - Seleção por torneio

A seleção por torneio escolhe alguns candidatos aleatoriamente e retorna o indivíduo com maior fitness entre eles. Esse operador mantém pressão seletiva sem depender de probabilidades proporcionais à fitness.


In [7]:
def tournament_selection(population, k=3): 
    candidates = random.sample(population, k)
    return max(candidates, key=lambda x: x["evaluation"])

#### Item 6.3 - Crossover uniforme

O crossover uniforme cria um novo indivíduo misturando genes de dois pais. Para cada hiperparâmetro, o valor é herdado de um dos pais com probabilidade igual.


In [8]:
def uniform_crossover(parent1, parent2, crossover_rate=0.85):
    if random.random() > crossover_rate:
        selected = random.choice([parent1, parent2])
        child = selected.copy()
        return child

    child = {}

    for key in parent1.keys():
        if random.random() < 0.5:
            child[key] = parent1[key]
        else:
            child[key] = parent2[key]

    return child

#### Item 6.4 - Mutação por gene

A mutação altera cada gene com uma probabilidade definida pelo experimento. Esse operador mantém diversidade na população e ajuda a explorar configurações que não apareceriam apenas por crossover.


In [9]:
def logreg_mutation(individual, mutation_rate=0.20):
    new = individual.copy()

    if random.random() < mutation_rate:
        new["max_iter"] = random.choice(MAX_ITER_SPACE)

    if random.random() < mutation_rate:
        new["C"] = random.choice(C_SPACE)

    if random.random() < mutation_rate:
        new["class_weight"] = random.choice(CLASS_WEIGHT_SPACE)

    if random.random() < mutation_rate:
        new["scaler_type"] = random.choice(SCALER_SPACE)

    return new


#### Item 6.5 - Nova geração com elitismo

A nova geração preserva os melhores indivíduos da geração atual e completa o restante da população com filhos gerados por seleção, crossover e mutação. O elitismo evita perder a melhor solução já encontrada.


In [10]:
def generate_new_generation(
    sorted_by_fitness_current_population,
    population_size,
    crossover_rate=0.85,
    mutation_rate=0.20,
    elite_size=2
):
    new_generation = []    

    
    elite = sorted_by_fitness_current_population[:elite_size]
    new_generation.extend([ind.copy() for ind in elite])

    while len(new_generation) < population_size:
        parent1 = tournament_selection(sorted_by_fitness_current_population)
        parent2 = tournament_selection(sorted_by_fitness_current_population)

        child = uniform_crossover(
            parent1,
            parent2,
            crossover_rate=crossover_rate
        )

        child = logreg_mutation(
            child,
            mutation_rate=mutation_rate
        )   
        new_generation.append(child)

    return new_generation

#### Item 6.6 - Execução do Algoritmo Genético com monitoramento

A função principal executa as gerações do Algoritmo Genético. Em cada geração, a população é avaliada, o melhor indivíduo é registrado e o histórico armazena informações para tracking de desempenho: melhor fitness, fitness média, desvio padrão, pior fitness, tempo da geração e configuração do experimento.

Esses registros funcionam como logging estruturado do treinamento e permitem comparar custo computacional e evolução da população entre diferentes configurações do AG.


In [11]:
def extract_fitness_values(population):
    return [individual["evaluation"] for individual in population]


def execute_ga(
    generation_count,
    population_size,
    crossover_rate,
    mutation_rate,
    elite_size,
    experiment_name="Experimento",
):
    current_generation = None
    history_data = []
    best_individuals = []
    experiment_started_at = time.perf_counter()

    for i in range(0, generation_count):
        generation_started_at = time.perf_counter()
        current_generation = generate_new_generation(
            sorted_by_fitness_current_population,
            population_size=population_size,
            crossover_rate=crossover_rate,
            mutation_rate=mutation_rate,
            elite_size=elite_size,
        ) if current_generation is not None else create_hot_start_individuals(population_size)

        sorted_by_fitness_current_population = evaluate_population(current_generation)
        best_evaluation = sorted_by_fitness_current_population[0]
        fitness_values = extract_fitness_values(sorted_by_fitness_current_population)
        generation_elapsed_seconds = time.perf_counter() - generation_started_at

        history_data.append({
            "Experimento": experiment_name,
            "Geração": i + 1,
            "População": population_size,
            "Taxa crossover": crossover_rate,
            "Taxa mutação": mutation_rate,
            "Elitismo": elite_size,
            "Melhor fitness": round(float(max(fitness_values)), 6),
            "Fitness média": round(float(np.mean(fitness_values)), 6),
            "Fitness desvio": round(float(np.std(fitness_values)), 6),
            "Pior fitness": round(float(min(fitness_values)), 6),
            "Tempo geração (s)": round(generation_elapsed_seconds, 3),
            "Melhor Indivíduo": best_evaluation.copy(),
        })
        best_individuals.append(best_evaluation.copy())

    top_individual = sorted(
        best_individuals,
        key=lambda item: item["evaluation"],
        reverse=True,
    )[0]

    experiment_elapsed_seconds = time.perf_counter() - experiment_started_at
    experiment_log = {
        "Experimento": experiment_name,
        "Gerações": generation_count,
        "População": population_size,
        "Taxa crossover": crossover_rate,
        "Taxa mutação": mutation_rate,
        "Elitismo": elite_size,
        "Melhor fitness": round(float(top_individual["evaluation"]), 6),
        "Tempo total (s)": round(experiment_elapsed_seconds, 3),
        "Tempo médio por geração (s)": round(experiment_elapsed_seconds / generation_count, 3),
        "Melhor Indivíduo": top_individual.copy(),
    }

    return pd.DataFrame(history_data), top_individual, experiment_log


#### Item 7 - Execução dos três experimentos e logs de desempenho

O trabalho exige pelo menos três experimentos com configurações diferentes do Algoritmo Genético. As execuções abaixo variam tamanho da população, taxa de mutação e número de gerações, mantendo o mesmo crossover e elitismo.

Além do melhor indivíduo, cada experimento gera logs estruturados para tracking de desempenho: fitness por geração, estatísticas da população e tempo de execução.


In [12]:
experiment_1_history, experiment_1_best, experiment_1_log = execute_ga(
    experiment_name="Experimento 1",
    generation_count=20,
    population_size=20,
    crossover_rate=0.85,
    mutation_rate=0.05,
    elite_size=2,
)

experiment_2_history, experiment_2_best, experiment_2_log = execute_ga(
    experiment_name="Experimento 2",
    generation_count=20,
    population_size=40,
    crossover_rate=0.85,
    mutation_rate=0.10,
    elite_size=2,
)

experiment_3_history, experiment_3_best, experiment_3_log = execute_ga(
    experiment_name="Experimento 3",
    generation_count=30,
    population_size=40,
    crossover_rate=0.85,
    mutation_rate=0.20,
    elite_size=2,
)

ga_experiment_summary = pd.DataFrame([
    experiment_1_log,
    experiment_2_log,
    experiment_3_log,
])

display(ga_experiment_summary)


,Experimento,Gerações,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Tempo total (s),Tempo médio por geração (s),Melhor Indivíduo
0,Experimento 1,20,20,0.85,0.05,2,0.279135,40.728,2.036,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
1,Experimento 2,20,40,0.85,0.10,2,0.279135,78.683,3.934,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
2,Experimento 3,30,40,0.85,0.20,2,0.279135,120.183,4.006,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"


#### Item 7.1 - Histórico e tracking do Experimento 1

Este histórico mostra o melhor indivíduo encontrado em cada geração do primeiro experimento, junto com métricas de monitoramento da população e tempo por geração.


In [13]:
display(experiment_1_history)

,Experimento,Geração,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Fitness média,Fitness desvio,Pior fitness,Tempo geração (s),Melhor Indivíduo
0,Experimento 1,1,20,0.85,0.05,2,0.259987,0.212432,0.071027,0.000000,4.152,"{'C': 1.0, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2599874074802453}"
1,Experimento 1,2,20,0.85,0.05,2,0.259987,0.240963,0.010996,0.232423,3.333,"{'C': 1.0, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2599874074802453}"
2,Experimento 1,3,20,0.85,0.05,2,0.259987,0.248648,0.012537,0.234778,2.624,"{'C': 1.0, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2599874074802453}"
3,Experimento 1,4,20,0.85,0.05,2,0.259987,0.242378,0.056379,0.000000,1.989,"{'C': 1.0, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2599874074802453}"
4,Experimento 1,5,20,0.85,0.05,2,0.259987,0.258728,0.005488,0.234807,1.775,"{'C': 1.0, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2599874074802453}"
5,Experimento 1,6,20,0.85,0.05,2,0.259987,0.259987,0.000000,0.259987,1.736,"{'C': 1.0, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2599874074802453}"
6,Experimento 1,7,20,0.85,0.05,2,0.279135,0.260945,0.004173,0.259987,1.726,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
7,Experimento 1,8,20,0.85,0.05,2,0.279135,0.248903,0.057389,0.000000,1.973,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
8,Experimento 1,9,20,0.85,0.05,2,0.279135,0.253690,0.058893,0.000000,1.718,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
9,Experimento 1,10,20,0.85,0.05,2,0.279135,0.276263,0.006837,0.259987,1.638,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"


#### Item 7.2 - Histórico e tracking do Experimento 2

Este histórico mostra o melhor indivíduo encontrado em cada geração do segundo experimento, junto com métricas de monitoramento da população e tempo por geração.


In [14]:
display(experiment_2_history)

,Experimento,Geração,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Fitness média,Fitness desvio,Pior fitness,Tempo geração (s),Melhor Indivíduo
0,Experimento 2,1,40,0.85,0.1,2,0.259987,0.205980,0.077954,0.000000,7.431,"{'C': 1.0, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2599874074802453}"
1,Experimento 2,2,40,0.85,0.1,2,0.259987,0.220270,0.063270,0.000000,6.468,"{'C': 1.0, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2599874074802453}"
2,Experimento 2,3,40,0.85,0.1,2,0.279135,0.246463,0.014148,0.234778,4.901,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
3,Experimento 2,4,40,0.85,0.1,2,0.279135,0.252305,0.043276,0.000000,3.834,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
4,Experimento 2,5,40,0.85,0.1,2,0.279135,0.238871,0.081223,0.000000,3.776,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
5,Experimento 2,6,40,0.85,0.1,2,0.279135,0.257728,0.060426,0.000000,3.552,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
6,Experimento 2,7,40,0.85,0.1,2,0.279135,0.242321,0.091841,0.000000,3.393,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
7,Experimento 2,8,40,0.85,0.1,2,0.279135,0.267042,0.044940,0.000000,3.480,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
8,Experimento 2,9,40,0.85,0.1,2,0.279135,0.258970,0.061278,0.000000,3.701,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
9,Experimento 2,10,40,0.85,0.1,2,0.279135,0.265897,0.044732,0.000000,3.541,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"


#### Item 7.3 - Histórico e tracking do Experimento 3

Este histórico mostra o melhor indivíduo encontrado em cada geração do terceiro experimento, junto com métricas de monitoramento da população e tempo por geração.


In [15]:
display(experiment_3_history)

,Experimento,Geração,População,Taxa crossover,Taxa mutação,Elitismo,Melhor fitness,Fitness média,Fitness desvio,Pior fitness,Tempo geração (s),Melhor Indivíduo
0,Experimento 3,1,40,0.85,0.2,2,0.276312,0.207622,0.078916,0.000000,7.390,"{'C': 0.1, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.27631169849669124}"
1,Experimento 3,2,40,0.85,0.2,2,0.276312,0.217666,0.073592,0.000000,6.465,"{'C': 0.1, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.27631169849669124}"
2,Experimento 3,3,40,0.85,0.2,2,0.279135,0.219501,0.093499,0.000000,4.389,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
3,Experimento 3,4,40,0.85,0.2,2,0.279135,0.261241,0.017621,0.232423,4.226,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
4,Experimento 3,5,40,0.85,0.2,2,0.279135,0.227638,0.096531,0.000000,3.629,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
5,Experimento 3,6,40,0.85,0.2,2,0.279135,0.233864,0.089748,0.000000,3.967,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
6,Experimento 3,7,40,0.85,0.2,2,0.279135,0.234617,0.090123,0.000000,3.664,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
7,Experimento 3,8,40,0.85,0.2,2,0.279135,0.242510,0.082413,0.000000,3.593,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
8,Experimento 3,9,40,0.85,0.2,2,0.279135,0.243067,0.082587,0.000000,3.572,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"
9,Experimento 3,10,40,0.85,0.2,2,0.279135,0.239807,0.082028,0.000000,4.096,"{'C': 0.01, 'class_weight': 'balanced', 'scaler_type': 'normalizer', 'max_iter': 2000, 'evaluation': 0.2791351369133341}"


#### Item 7.4 - Validação cruzada do melhor indivíduo do Experimento 1

A célula abaixo reexibe as métricas médias de validação cruzada para o melhor indivíduo encontrado no primeiro experimento.


In [16]:
execute_logistic_regression(
    x=X_train_sample,
    y=y_train_sample,
    C=experiment_1_best["C"],
    class_weight=experiment_1_best["class_weight"],
    scaler_type=experiment_1_best["scaler_type"],
    max_iter=experiment_1_best["max_iter"],
    show=True,
)


########### LogisticRegression ##############
C: 0.01
class_weight: balanced
solver: liblinear
scaler_type: normalizer
max_iter: 2000
accuracy: 0.44418749999999996
recall: 0.7039978213507626
f1: 0.09705112929729327
precision: 0.05211872380691475
roc_auc: 0.6080401644926156
balanced_accuracy: 0.5683362321564337
#############################################


ClassificationMetrics(accuracy=0.44418749999999996, recall=0.7039978213507626, f1=0.09705112929729327, precision=0.05211872380691475, roc_auc=0.6080401644926156, balanced_accuracy=0.5683362321564337)

#### Item 7.5 - Validação cruzada do melhor indivíduo do Experimento 2

A célula abaixo reexibe as métricas médias de validação cruzada para o melhor indivíduo encontrado no segundo experimento.


In [17]:
execute_logistic_regression(
    x=X_train_sample,
    y=y_train_sample,
    C=experiment_2_best["C"],
    class_weight=experiment_2_best["class_weight"],
    scaler_type=experiment_2_best["scaler_type"],
    max_iter=experiment_2_best["max_iter"],
    show=True,
)


########### LogisticRegression ##############
C: 0.01
class_weight: balanced
solver: liblinear
scaler_type: normalizer
max_iter: 2000
accuracy: 0.44418749999999996
recall: 0.7039978213507626
f1: 0.09705112929729327
precision: 0.05211872380691475
roc_auc: 0.6080401644926156
balanced_accuracy: 0.5683362321564337
#############################################


ClassificationMetrics(accuracy=0.44418749999999996, recall=0.7039978213507626, f1=0.09705112929729327, precision=0.05211872380691475, roc_auc=0.6080401644926156, balanced_accuracy=0.5683362321564337)

#### Item 7.6 - Validação cruzada do melhor indivíduo do Experimento 3

A célula abaixo reexibe as métricas médias de validação cruzada para o melhor indivíduo encontrado no terceiro experimento.


In [18]:
execute_logistic_regression(
    x=X_train_sample,
    y=y_train_sample,
    C=experiment_3_best["C"],
    class_weight=experiment_3_best["class_weight"],
    scaler_type=experiment_3_best["scaler_type"],
    max_iter=experiment_3_best["max_iter"],
    show=True,
)


########### LogisticRegression ##############
C: 0.01
class_weight: balanced
solver: liblinear
scaler_type: normalizer
max_iter: 2000
accuracy: 0.44418749999999996
recall: 0.7039978213507626
f1: 0.09705112929729327
precision: 0.05211872380691475
roc_auc: 0.6080401644926156
balanced_accuracy: 0.5683362321564337
#############################################


ClassificationMetrics(accuracy=0.44418749999999996, recall=0.7039978213507626, f1=0.09705112929729327, precision=0.05211872380691475, roc_auc=0.6080401644926156, balanced_accuracy=0.5683362321564337)

#### Item 8 - Avaliação final da Regressão Logística otimizada

Depois dos três experimentos, o melhor indivíduo é escolhido pela maior fitness média. O modelo final é treinado uma única vez com a amostra de treino e avaliado na amostra estratificada de teste.

Nesta etapa não há validação cruzada: o teste é usado apenas para medir o desempenho final do modelo otimizado pelo Algoritmo Genético.


In [19]:
best_individual = max([experiment_1_best, experiment_2_best, experiment_3_best], key=lambda x: x["evaluation"])

final_model = LogisticRegression(
    C=best_individual["C"],
    class_weight=best_individual["class_weight"],
    solver="liblinear",
    max_iter=best_individual["max_iter"],
    random_state=RANDOM_STATE,
)

X_train_final, X_test_final = apply_scaler(
    X_train_sample,
    X_test_sample,
    best_individual["scaler_type"],
)

final_model.fit(X_train_final, y_train_sample)
y_pred = final_model.predict(X_test_final)
y_score = final_model.predict_proba(X_test_final)[:, 1]
current_execution = calculate_metrics(y_test_sample, y_pred, y_score)
display(current_execution)


ClassificationMetrics(accuracy=0.444125, recall=0.696165191740413, f1=0.09595446228908315, precision=0.05152838427947598, roc_auc=0.6163168313324315, balanced_accuracy=0.5645686942907783)

#### Item 9 - Comparação com o baseline de Regressão Logística da Parte 6

A comparação abaixo usa como referência a Regressão Logística tunada com `GridSearchCV` na Parte 6. Como a base é desbalanceada, a leitura principal deve priorizar F1, recall, precision e balanced accuracy, e não apenas accuracy.

O modelo otimizado por Algoritmo Genético é avaliado com o melhor indivíduo encontrado nos três experimentos anteriores, usando uma amostra estratificada de teste com 8.000 registros, mesma escala usada no baseline da Parte 6.


In [20]:
baseline_part_6_metrics = {
    "modelo": "Regressão Logística baseline Parte 6 (GridSearchCV)",
    "accuracy": 0.6784,
    "balanced_accuracy": 0.5924,
    "precision": 0.0657,
    "recall": 0.4985,
    "f1": 0.1161,
    "roc_auc": 0.6348,
}

optimized_ga_metrics = {
    "modelo": "Regressão Logística otimizada com Algoritmo Genético",
    "accuracy": round(current_execution.accuracy, 4),
    "balanced_accuracy": round(current_execution.balanced_accuracy, 4),
    "precision": round(current_execution.precision, 4),
    "recall": round(current_execution.recall, 4),
    "f1": round(current_execution.f1, 4),
    "roc_auc": round(current_execution.roc_auc, 4),
}

metrics_comparison = pd.DataFrame([
    baseline_part_6_metrics,
    optimized_ga_metrics,
])

metrics_delta = {
    "modelo": "Diferença AG - baseline",
    "accuracy": round(optimized_ga_metrics["accuracy"] - baseline_part_6_metrics["accuracy"], 4),
    "balanced_accuracy": round(optimized_ga_metrics["balanced_accuracy"] - baseline_part_6_metrics["balanced_accuracy"], 4),
    "precision": round(optimized_ga_metrics["precision"] - baseline_part_6_metrics["precision"], 4),
    "recall": round(optimized_ga_metrics["recall"] - baseline_part_6_metrics["recall"], 4),
    "f1": round(optimized_ga_metrics["f1"] - baseline_part_6_metrics["f1"], 4),
    "roc_auc": round(optimized_ga_metrics["roc_auc"] - baseline_part_6_metrics["roc_auc"], 4),
}

metrics_comparison = pd.concat(
    [metrics_comparison, pd.DataFrame([metrics_delta])],
    ignore_index=True,
)

metrics_comparison


,modelo,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
0,Regressão Logística baseline Parte 6 (GridSearchCV),0.6784,0.5924,0.0657,0.4985,0.1161,0.6348
1,Regressão Logística otimizada com Algoritmo Genético,0.4441,0.5646,0.0515,0.6962,0.0960,0.6163
2,Diferença AG - baseline,-0.2343,-0.0278,-0.0142,0.1977,-0.0201,-0.0185


#### Item 9.1 - Relatório final da Regressão Logística otimizada

O relatório de classificação e a matriz de confusão detalham o comportamento do modelo final na amostra de teste. Esses resultados ajudam a interpretar o trade-off entre recuperar mais casos positivos e manter precisão aceitável.


In [21]:
final_classification_report = pd.DataFrame(
    classification_report(
        y_test_sample,
        y_pred,
        output_dict=True,
        zero_division=0,
    )
).T

final_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_test_sample, y_pred),
    index=["real_0", "real_1"],
    columns=["pred_0", "pred_1"],
)

display(final_classification_report)
display(final_confusion_matrix)


,precision,recall,f1-score,support
0,0.969883,0.432972,0.598682,7661.000000
1,0.051528,0.696165,0.095954,339.000000
accuracy,0.444125,0.444125,0.444125,0.444125
macro avg,0.510706,0.564569,0.347318,8000.000000
weighted avg,0.930968,0.444125,0.577379,8000.000000


,pred_0,pred_1
real_0,3317,4344
real_1,103,236


#### Item 9.2 - Conclusão da otimização com Algoritmo Genético

A conclusão abaixo sintetiza a melhor configuração encontrada pelo Algoritmo Genético, compara o resultado com o baseline da Parte 6 e registra a interpretação das métricas mais importantes para a classe positiva.


In [22]:
f1_delta = metrics_delta["f1"]
recall_delta = metrics_delta["recall"]
precision_delta = metrics_delta["precision"]
balanced_accuracy_delta = metrics_delta["balanced_accuracy"]

conclusion_rows = [
    {
        "aspecto": "melhor_individuo_ag",
        "conclusao": f"C={best_individual['C']}, class_weight={best_individual['class_weight']}, scaler_type={best_individual['scaler_type']}, max_iter={best_individual['max_iter']}, fitness={best_individual['evaluation']:.4f}.",
    },
    {
        "aspecto": "comparacao_f1",
        "conclusao": f"O F1 da Regressão Logística com AG foi {optimized_ga_metrics['f1']:.4f}, contra {baseline_part_6_metrics['f1']:.4f} no baseline da Parte 6. Diferença: {f1_delta:+.4f}.",
    },
    {
        "aspecto": "comparacao_recall",
        "conclusao": f"O recall da Regressão Logística com AG foi {optimized_ga_metrics['recall']:.4f}, contra {baseline_part_6_metrics['recall']:.4f} no baseline. Diferença: {recall_delta:+.4f}.",
    },
    {
        "aspecto": "comparacao_precision",
        "conclusao": f"A precision da Regressão Logística com AG foi {optimized_ga_metrics['precision']:.4f}, contra {baseline_part_6_metrics['precision']:.4f} no baseline. Diferença: {precision_delta:+.4f}.",
    },
    {
        "aspecto": "balanced_accuracy",
        "conclusao": f"A balanced accuracy da Regressão Logística com AG foi {optimized_ga_metrics['balanced_accuracy']:.4f}, contra {baseline_part_6_metrics['balanced_accuracy']:.4f} no baseline. Diferença: {balanced_accuracy_delta:+.4f}.",
    },
    {
        "aspecto": "interpretacao",
        "conclusao": "Como se trata de um problema de saúde, a redução de falsos negativos é importante. O resultado deve ser lido como trade-off entre recall, precision, F1 e balanced accuracy, sem usar accuracy isolada como critério principal.",
    },
    {
        "aspecto": "decisao_final",
        "conclusao": "A Regressão Logística otimizada por Algoritmo Genético deve ser comparada ao baseline da Parte 6 e aos demais modelos otimizados. A decisão final depende do equilíbrio entre sensibilidade para a classe positiva e volume de falsos positivos.",
    },
]

optimization_conclusion = pd.DataFrame(conclusion_rows)
optimization_conclusion


,aspecto,conclusao
0,melhor_individuo_ag,"C=0.01, class_weight=balanced, scaler_type=normalizer, max_iter=2000, fitness=0.2791."
1,comparacao_f1,"O F1 da Regressão Logística com AG foi 0.0960, contra 0.1161 no baseline da Parte 6. Diferença: -0.0201."
2,comparacao_recall,"O recall da Regressão Logística com AG foi 0.6962, contra 0.4985 no baseline. Diferença: +0.1977."
3,comparacao_precision,"A precision da Regressão Logística com AG foi 0.0515, contra 0.0657 no baseline. Diferença: -0.0142."
4,balanced_accuracy,"A balanced accuracy da Regressão Logística com AG foi 0.5646, contra 0.5924 no baseline. Diferença: -0.0278."
5,interpretacao,"Como se trata de um problema de saúde, a redução de falsos negativos é importante. O resultado deve ser lido como trade-off entre recall, precision, F1 e balanced accuracy, sem usar accuracy isolada como critério principal."
6,decisao_final,A Regressão Logística otimizada por Algoritmo Genético deve ser comparada ao baseline da Parte 6 e aos demais modelos otimizados. A decisão final depende do equilíbrio entre sensibilidade para a classe positiva e volume de falsos positivos.


---

## Resultado da Parte 8

Este notebook implementa um Algoritmo Genético para otimizar hiperparâmetros da Regressão Logística (`LogisticRegression`) usando as bases encoded finais da Parte 2.

A avaliação dos indivíduos ocorre por validação cruzada estratificada com 5 folds, usando uma fitness voltada à classe positiva:

```text
fitness = 0.7 * F1 + 0.3 * recall
```

Foram definidos três experimentos com diferentes configurações de população, mutação e gerações. Ao final, o melhor indivíduo entre os experimentos é usado para treinar um modelo final e avaliar seu desempenho na amostra de teste.

A seção final compara a Regressão Logística otimizada por Algoritmo Genético com o baseline de Regressão Logística da Parte 6, usando a mesma escala de amostra de teste. A interpretação prioriza métricas adequadas para desbalanceamento, especialmente recall, F1, precision e balanced accuracy.

A conclusão operacional deve ser lida a partir da tabela `optimization_conclusion`. Se o recall do modelo com AG ficar acima do baseline da Parte 6, isso indica maior sensibilidade para capturar positivos. Caso isso venha acompanhado de queda em precision, F1 ou balanced accuracy, o resultado deve ser apresentado como um trade-off esperado em triagem de saúde: maior sensibilidade para capturar positivos, com mais falsos positivos e necessidade de confirmação diagnóstica posterior.

O monitoramento do AG é registrado em dois níveis: `experiment_1_history`, `experiment_2_history` e `experiment_3_history` armazenam a evolução por geração; `ga_experiment_summary` consolida a configuração, o melhor fitness e o tempo total de cada experimento. Esses logs permitem rastrear desempenho, convergência e custo computacional entre as configurações testadas.

